# CNN Model Development - NIH Chest X-ray Classification

### Objective

This notebook develops a simple Convolutional Neural Network (CNN) using PyTorch to classify chest X-ray images into two categories: Effusion and No Finding.

### Execution Plan

In this notebook, we will
1. Load existing dataset and construct PyTorch DataLoaders
2. Build CNN architecture
3. Inspect model parameters and tensor dimensions
4. Train CNN using native PyTorch training loop
5. Evaluate model performance on validation and test sets
6. Compare CNN against MLP
7. Experiment with image augmentation and assess its impact on model performance

### Learning Focus

Understand how convolutional layers, feature maps, pooling, and fully connected layers work together to extract image features and perform classification. 

### Limitations

This is an educational project and is not intended for clinical diagnosis.

## 01. Dataset

NIH ChestX-ray14 subset of 3241 images, will be split following the MLP model for consistency and to allow for comparisons between the 2 models.

In [13]:
import torch
import pandas as pd

# Load preprocessed tensors
data = torch.load(
    "../../data/processed/mlp_images_64x64.pt",
    map_location="cpu",
    weights_only=True)

X = data["X"]
y = data["y"]
image_names = data["image_names"]
patient_ids = data["patient_ids"]  
splits = data["splits"]  

print("Images:", X.shape)
print("Labels:", y.shape)
print("Number of patients:", len(patient_ids))
print("Number of splits:", len(splits))
print("Data type:", X.dtype)
print("Minimum:", X.min().item())
print("Maximum:", X.max().item())

# Load dataset metadata
df = pd.read_csv("../../data/processed/mlp_dataset.csv")
print("Columns:", df.columns.tolist())
print(f"Split distribution: {df.groupby(['Split', 'Label']).size()}")


Images: torch.Size([3241, 1, 64, 64])
Labels: torch.Size([3241])
Number of patients: 3241
Number of splits: 3241
Data type: torch.float32
Minimum: 0.0
Maximum: 1.0
Columns: ['Image Index', 'Patient ID', 'Label', 'Split']
Split distribution: Split  Label
test   0         329
       1         150
train  0        1939
       1         269
val    0         486
       1          68
dtype: int64


## 02. Data Preparation

We will confirm that data for CNN model development is identical to the data used for MLP model development:
- 3241 images, with matching tensor and CSV 
- 2208 training, 554 validation, and 479 test images
- Labels encoded as 0 and 1
- Original patient-level splits preserved
- No need to preprocess or split data again

In [15]:
from collections import Counter

# Inspect split and label values
print("Split values:", Counter(splits))
print("Label values:", Counter(y.tolist()))

print("First 5 image names:", image_names[:5])
print("First 5 patient IDs:", patient_ids[:5])

# Verify CSV metadata
print("CSV columns:", df.columns.tolist())
print("Split and label distribution:")
print(df.groupby(["Split", "Label"]).size())
print("First 5 rows:")
display(df.head())

# Check tensor and CSV ordering match
print("Tensor images:", len(X))
print("CSV rows:", len(df))

print("Image names match:",
      image_names == df["Image Index"].tolist())

Split values: Counter({'train': 2208, 'val': 554, 'test': 479})
Label values: Counter({0: 2754, 1: 487})
First 5 image names: ['00000002_000.png', '00000005_000.png', '00000005_001.png', '00000005_002.png', '00000005_003.png']
First 5 patient IDs: [2, 5, 5, 5, 5]
CSV columns: ['Image Index', 'Patient ID', 'Label', 'Split']
Split and label distribution:
Split  Label
test   0         329
       1         150
train  0        1939
       1         269
val    0         486
       1          68
dtype: int64
First 5 rows:


,Image Index,Patient ID,Label,Split
0,00000002_000.png,2,0,train
1,00000005_000.png,5,0,train
2,00000005_001.png,5,0,train
3,00000005_002.png,5,0,train
4,00000005_003.png,5,0,train


Tensor images: 3241
CSV rows: 3241
Image names match: True


## 03. Model Architecture

### 3.1 Creating PyTorch Datasets and DataLoaders

Datasets store and provide access to individual sample and their labels.
DataLoader retrieves samples from the Dataset and groups them into batches. Training will be performed on these manageable subsets of data.

In [17]:
from torch.utils.data import TensorDataset, DataLoader, Subset

# Create a dataset containing images and labels
dataset = TensorDataset(X, y)

# Get indices for each split
train_idx = [i for i, s in enumerate(splits) if s == "train"]
val_idx = [i for i, s in enumerate(splits) if s == "val"]
test_idx = [i for i, s in enumerate(splits) if s == "test"]

# Create separate datasets
train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
test_dataset = Subset(dataset, test_idx)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

# Inspect 1 batch
images, labels = next(iter(train_loader))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)
print("Labels in batch:", labels.unique())

Training batches: 69
Validation batches: 18
Test batches: 15
Image batch shape: torch.Size([32, 1, 64, 64])
Label batch shape: torch.Size([32])
Image dtype: torch.float32
Label dtype: torch.int64
Labels in batch: tensor([0, 1])


There are 69 training batches, 18 validation batches, and 15 test batches of 32 images each. 

Each image has 1 channel (grayscale), 64 pixels in height and 64 pixels in width (64x64), with the labels containing corresponding diagnosis class for each image.

### 3.2 Building the CNN Architecture

1. First convolution layer (nn.Conv2d)
2. ReLU activation to introduce non-linearity
3. MaxPooling (nn.MaxPool)
4. Second convolution layer (nn.Conv2d)
5. ReLU activation
6. MaxPooling
7. nn.Flatten
8. nn.Linear

In [29]:
import torch.nn as nn

# CNN Model
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Flatten(),
    nn.Linear(8192, 2)) #(32*16*16)

## 04. Training CNN model

### 4.1 Configure Loss Function, Optimizer, and Device

In [30]:
import torch.optim as optim

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
model = model.to(device)

# Loss Function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

Using device: cpu


### 4.2 Training Loop

In [31]:
NUM_EPOCHS = 10
cnn_train_losses = []
cnn_val_losses = []

for epoch in range(NUM_EPOCHS):
    # Training
    model.train()
    running_train_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()
    epoch_train_loss = running_train_loss / len(train_loader)
    cnn_train_losses.append(epoch_train_loss)

    # Validation
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()
    epoch_val_loss = running_val_loss / len(val_loader)
    cnn_val_losses.append(epoch_val_loss)

    # Results
    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
        f"Train Loss: {epoch_train_loss:.4f} "
        f"Val Loss: {epoch_val_loss:.4f}"
    )            

Epoch [1/10] Train Loss: 0.3797 Val Loss: 0.3859
Epoch [2/10] Train Loss: 0.3440 Val Loss: 0.3422
Epoch [3/10] Train Loss: 0.3276 Val Loss: 0.3443
Epoch [4/10] Train Loss: 0.3275 Val Loss: 0.3336
Epoch [5/10] Train Loss: 0.3101 Val Loss: 0.3239
Epoch [6/10] Train Loss: 0.3034 Val Loss: 0.3528
Epoch [7/10] Train Loss: 0.2983 Val Loss: 0.3328
Epoch [8/10] Train Loss: 0.2885 Val Loss: 0.3411
Epoch [9/10] Train Loss: 0.2893 Val Loss: 0.3454
Epoch [10/10] Train Loss: 0.2806 Val Loss: 0.3576
